# 🧠 qwen3_5 実装 × sm75

線形注意(GDN)・flash_attn・バックエンド選択を wheel から確認。


In [ ]:
import glob
import pathlib
import subprocess
import sys
import zipfile

subprocess.run([sys.executable, "-m", "pip", "download", "--no-cache-dir",
                "--no-deps", "vllm==0.32.0", "-d", "/tmp/w"],
               capture_output=True, text=True)
whl = glob.glob("/tmp/w/*.whl")
print("wheel:", whl)
if not whl:
    raise SystemExit("wheel 取得失敗")
zipfile.ZipFile(whl[0]).extractall("/tmp/vl")
root = pathlib.Path("/tmp/vl/vllm")
print("py files:", len(list(root.rglob("*.py"))))

print("\n=== qwen3_5 関連ファイル ===")
targets = []
for f in sorted(root.rglob("qwen3_5*.py")):
    print("  ", f.relative_to(root), f.stat().st_size)
    targets.append(f)
for pat in ("*mamba*.py", "*linear_attn*.py", "*gdn*.py"):
    for f in sorted(root.rglob(pat)):
        print("  ", f.relative_to(root), f.stat().st_size)
        targets.append(f)

mf = root / "model_executor/models/qwen3_5.py"
txt = mf.read_text(errors="replace") if mf.exists() else ""
print("\n=== qwen3_5.py の import ===")
for i, l in enumerate(txt.splitlines(), 1):
    s = l.strip()
    if s.startswith("import ") or s.startswith("from "):
        print(f"  {i}: {s[:150]}")

print("\n=== qwen3_5.py の層構築（linear_attn / MTP）===")
for i, l in enumerate(txt.splitlines(), 1):
    if any(k in l for k in ("linear_attn", "LinearAttention", "mamba",
                            "mtp", "MTP", "gdn", "conv1d")):
        print(f"  {i}: {l.strip()[:160]}")

print("\n=== 危険シンボル（sm75 で落ちうる）===")
DANGER = ["flash_attn", "causal_conv1d", "fla.ops", "from fla", "triton",
          "chunk_gated", "fused_recurrent", "selective_scan", "cutlass",
          "flashinfer"]
for f in sorted(set(targets)):
    t = f.read_text(errors="replace")
    hits = [k for k in DANGER if k in t]
    if hits:
        print(f"  {f.relative_to(root)}: {hits}")

print("\n=== capability / sm 判定がある行（対象ファイル内）===")
for f in sorted(set(targets)):
    t = f.read_text(errors="replace")
    for i, l in enumerate(t.splitlines(), 1):
        s = l.strip()
        if ("capability" in s or "is_sm" in s or "major" in s
                or "75" in s or "80" in s or "90" in s) and (
                ">=" in s or "<" in s or "==" in s or "if " in s):
            print(f"  {f.relative_to(root)}:{i}: {s[:150]}")

print("\n=== attention backend の優先順位（T4 はここで決まる）===")
sel = list(root.rglob("attention/selector.py")) + list(root.rglob("attention/backends/registry.py"))
for f in sel:
    print(f"--- {f.relative_to(root)} ---")
    t = f.read_text(errors="replace")
    for i, l in enumerate(t.splitlines(), 1):
        s = l.strip()
        if any(k in s for k in ("priority", "backend", "capability", "major",
                                "def get_attn_backend")) and (
                "=" in s or "def " in s or "if " in s):
            print(f"  {i}: {s[:150]}")

cu = root / "platforms/cuda.py"
if cu.exists():
    t = cu.read_text(errors="replace")
    i = t.find("def _get_backend_priority")
    print("\n=== cuda._get_backend_priority ===")
    print(t[i:i + 2200] if i > 0 else "関数が見つからない")
